In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# LOAD DATASET
df = pd.read_csv("../merged_dataset.csv")
df.head()

## Target Selection & HARD SCALE FIX

In [ ]:
TARGET = "FinalGrade"

if TARGET not in df.columns:
    raise ValueError("FinalGrade column not found")

print("Before scaling:", df[TARGET].min(), df[TARGET].max())

# HARD LOCK: this notebook only works with 0–100 target
if df[TARGET].max() == 3:
    df[TARGET] = df[TARGET] * (100 / 3)
elif df[TARGET].max() <= 1:
    df[TARGET] = df[TARGET] * 100

print("After scaling:", df[TARGET].min(), df[TARGET].max())

if df[TARGET].max() <= 10:
    raise ValueError("FinalGrade scale is still wrong. Expected 0–100.")

## Data Cleaning, Encoding & Leakage Removal

In [ ]:
df2 = df.dropna(subset=[TARGET]).copy()

num_cols = df2.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df2.columns if c not in num_cols]

for c in num_cols:
    df2[c] = df2[c].fillna(df2[c].median())

for c in cat_cols:
    df2[c] = df2[c].fillna(df2[c].mode()[0])
    if c != TARGET:
        df2[c] = LabelEncoder().fit_transform(df2[c].astype(str))

# REMOVE DATA LEAKAGE
X = df2.drop(columns=[TARGET, "ExamScore"])
y = df2[TARGET]

print("Features used:", X.columns.tolist())
print("Target range:", y.min(), y.max())

##  Data Analysis (Graphs)

In [ ]:
import seaborn as sns

corr = df.drop(columns=[TARGET]).corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap="coolwarm",
    vmin=-0.5, vmax=0.5,
    square=True,
    linewidths=0.5,
    annot=False,
    cbar_kws={"shrink": 0.8}
)

plt.title("Correlation Matrix of Student Features", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## Train Multiple Regression Models

In [ ]:
def eval_reg(y_true, y_pred):
    return (
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        r2_score(y_true, y_pred)
    )

models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "DecisionTree": DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=7))
    ])
}

results, preds = [], {}

for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    preds[name] = p
    mae, rmse, r2 = eval_reg(y_test, p)
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})

results_df = pd.DataFrame(results).sort_values("MAE")
results_df.to_csv(OUTPUT_DIR / "model_evaluation.csv", index=False)
results_df

## Model Comparison Graphs

In [ ]:
for metric in ["MAE", "RMSE", "R2"]:
    plt.figure()
    plt.bar(results_df["Model"], results_df[metric])
    plt.title(f"{metric} Comparison")
    plt.ylabel(metric)
    plt.savefig(OUTPUT_DIR / f"{metric.lower()}_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## Actual vs Predicted (Best Model)

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_pred = preds[best_name]

plt.scatter(y_test, best_pred, alpha=0.7)
plt.xlabel("Actual Final Grade")
plt.ylabel("Predicted Final Grade")
plt.title(f"Actual vs Predicted ({best_name})")
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

## Feature Importance (Decision Tree)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
import pandas as pd
import matplotlib.pyplot as plt

dt = models["DecisionTree"]
dt.fit(X_train, y_train)

fi = pd.Series(dt.feature_importances_, index=X_train.columns).sort_values(ascending=False)

top_n = 10
fi_top = fi.head(top_n)

plt.figure(figsize=(10, 6))
plt.barh(fi_top.index[::-1], fi_top.values[::-1])
plt.xlabel("Importance")
plt.title(f"Top {top_n} Feature Importances (Decision Tree)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


## Single Student Prediction

In [ ]:
student = {}
for col in X.columns:
    default = round(float(X[col].median()), 2)
    val = input(f"{col} (default {default}): ").strip()
    student[col] = float(val) if val else default

row = pd.DataFrame([student])
pred = best_model.predict(row)[0]
pred = float(np.clip(pred, 0, 100))

print("Predicted Final Grade =", round(pred, 2))